In [ ]:
from homeharvest import scrape_property
from dotenv import load_dotenv
import os

load_dotenv()

os.getenv('USER')

df = scrape_property(
    location="Ann Arbor, MI",
    listing_type="for_rent",  # for_sale, for_rent, pending
    past_days=500)

drop_cols = ['office_phones', 'agent_phones', 'primary_photo', 'alt_photos', 'text']
df = df.drop(drop_cols, axis =1 )


In [6]:
import pandas

In [6]:
from homeharvest import scrape_property
from dotenv import load_dotenv
import os
import sqlalchemy
import pandas

load_dotenv()

engine = sqlalchemy.create_engine(f"postgresql://{os.getenv('USER')}:{os.getenv('PASS')}@{os.getenv('HOST')}:{os.getenv('port')}/{os.getenv('database')}")
con = engine.connect()


#rent = pandas.read_sql('SELECT * from raw.realtor__for_rent', con=con)

In [1]:
from utils.database_utils import query_last_updated

query_last_updated('raw.realtor__for_rent', 'last_update_date')

Timestamp('2026-05-03 08:44:22')

In [8]:
con = engine.connect()

rent.to_sql('realtor__for_rent', con=con, chunksize=500, schema='raw',if_exists='replace', index=False)

364

In [2]:
from homeharvest import scrape_property
from pandas import Timestamp
from utils.database_utils import query_last_updated, postgres_engine
from datetime import time
import pandas

def scrape_for_rent():

    con = postgres_engine()

    df = scrape_property(
        location="Ann Arbor, MI",
        listing_type="for_rent",  # for_sale, for_rent, pending
        past_days=100)

    drop_cols = ['office_phones', 'agent_phones', 'primary_photo', 'alt_photos', 'text']
    df = df.drop(drop_cols, axis =1 )

    df.loc[:, 'data_loaded_at'] = Timestamp('now')

    #query updated for filter
    query_time = query_last_updated(
        'raw.realtor__for_rent', 
        'last_update_date',
        con = con)
    
    query_time = query_time.date()

    df = df[df.last_update_date.dt.date > query_time]
    con.close()
    return df


In [3]:
d = scrape_for_rent()

In [15]:
def sqlite_keep_alive():
    # pip install sqlitecloud
    import os
    import sqlitecloud
    from dotenv import load_dotenv
    load_dotenv()

    # Open the connection to SQLite Cloud
    conn = sqlitecloud.connect(f"{os.getenv('SQLITE_CLOUD_URI')}{os.getenv('SQLITE_CLOUD_KEY')}")
    cursor = conn.execute("SELECT name FROM sqlite_master WHERE type='table';")
    result = cursor.fetchone()
    conn.close()

    print(result)

In [16]:
sqlite_keep_alive()

('_sqliteai_vector',)
